# GEC on one recogniser's transcript: few-shot, Gemini

The LLM corrects one recogniser's transcript at a time (Whisper-large-v3, Omnilingual LLM 1B v2, Indic Transcribe Core), on all 2,400 clips. Prompt `gec_few_shot` in `PROMPTS.md`.

Each corrected transcript is scored against the same recogniser's baseline transcript, per dataset subset, on all clips and on references with 3+ words.

## 1. Setup

In [1]:
FOLDER = "02_gec_single_recogniser"
NAME = "gec_few_shot_gemini"
LIMIT = None
WORKERS = 12
CHUNK = 200

import os, re, json, time, string, unicodedata, platform, importlib.metadata
os.chdir('/mnt/d/Sajjad/08-2023/Python Code/FYP26-27 Qwen Urdu ASRs+GEC/Urdu-ASR/experiments/02_gec_single_recogniser')
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, jiwer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA = ROOT / "data"
RESULTS = ROOT / "experiments" / FOLDER / "results"
for sub in ("corrections", "scores", "run_info"):
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)

RECOGNISERS = {
    "whisper_large_v3": "Whisper-large-v3",
    "omnilingual_llm_1b_v2": "Omnilingual LLM 1B v2",
    "indic_transcribe_core": "Indic Transcribe Core",
}
SHORT = {"whisper_large_v3": "Whisper", "omnilingual_llm_1b_v2": "Omni", "indic_transcribe_core": "Indic"}

# clean_urdu v1
_DIAC_RE = re.compile("[ؐ-ؚـً-ٰۖ-ۜ۟-۪ۨ-ۭ]")
_PUNCT = set(string.punctuation) | set("۔،؟؛٪।॥“”"
                                       "‘’—–…«»‖‐‒·")
_IZAFAT = {"اے", "ای", "ئے", "ء", "ے", "او"}
_UMAP = {"ي": "ی", "ك": "ک", "ة": "ہ", "ۃ": "ہ",
         "ى": "ی", "إ": "ا", "أ": "ا", "آ": "ا",
         "ٲ": "ا", "ٳ": "ا", "ؤ": "و", "ئ": "ی"}

def clean_urdu(text):
    t = unicodedata.normalize("NFC", str(text))
    t = _DIAC_RE.sub("", t)
    t = "".join(_UMAP.get(ch, ch) for ch in t)
    t = "".join(" " if ch in _PUNCT else ch for ch in t)
    return " ".join(w for w in t.split() if w not in _IZAFAT).strip()


def prompt_text(text):
    t = _DIAC_RE.sub("", unicodedata.normalize("NFC", str(text)))
    t = "".join(" " if ch in _PUNCT else ch for ch in t)
    return " ".join(t.split())


by_clip = pd.read_csv(DATA / "transcripts" / "all_recognisers_by_clip.csv", keep_default_na=False)
assert len(by_clip) == 2400, len(by_clip)
work_clips = by_clip.sample(LIMIT, random_state=42).sort_index() if LIMIT else by_clip
print(f"{len(by_clip)} clips; this run covers {len(work_clips)}")

examples = pd.read_csv(DATA / "few_shot_examples.csv", keep_default_na=False)
assert len(examples) == 5
assert not set(examples.clip_id) & set(by_clip.clip_id), "example clips must not be scored clips"
print("Few-shot example clips:", ", ".join(examples.clip_id))

2400 clips; this run covers 2400
Few-shot example clips: fleurs/test/17944002793012079254, fleurs/test/1144630914630122970, fleurs/test/5871438281700834673, fleurs/test/1136035778246969234, fleurs/validation/8466778121606878841


## 2. LLM

In [2]:
import os
import time
from openai import OpenAI

# Note: OpenRouter requires provider prefixes for model slugs (e.g., "google/gemini-...")
GEMINI_MODEL = "google/gemini-3.1-flash-lite"  # Adjust slug as needed on OpenRouter
LLM_NAME = GEMINI_MODEL


def load_env(path):
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env(ROOT / ".env")

if not os.environ.get("OPENROUTER_API_KEY"):
    raise RuntimeError("OPENROUTER_API_KEY is not set. Put OPENROUTER_API_KEY=your-key in a file named .env in the repository root.")

# Initialize the OpenAI client pointing to OpenRouter's endpoint
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)


def call_llm(prompt, temperature=0.0, max_tokens=512):
    last = None
    for attempt in range(6):
        try:
            resp = client.chat.completions.create(
                model=GEMINI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            
            usage = resp.usage
            choice = resp.choices[0]
            cut = choice.finish_reason == "length"
            
            return {
                "text": choice.message.content or "", 
                "tokens_in": usage.prompt_tokens or 0,
                "tokens_out": usage.completion_tokens or 0, 
                "cut": cut
            }
        except Exception as e:
            last = e
            time.sleep(min(2 ** (attempt + 1), 60))
            
    raise RuntimeError(f"{type(last).__name__}: {str(last)[:150]}")

## 3. Prompt and correction

In [3]:
def one_line(text):
    for line in str(text).strip().splitlines():
        line = line.strip()
        if line:
            return re.sub(r"^(Corrected|Output)\s*:\s*", "", line).strip()
    return ""


def correct(prompt, fallback):
    t0 = time.perf_counter()
    try:
        r = call_llm(prompt)
    except Exception as e:
        return {"output": fallback, "status": f"error: {e}", "llm_seconds": round(time.perf_counter() - t0, 2),
                "calls": 1, "tokens_in": 0, "tokens_out": 0}
    out = one_line(r["text"])
    status = "ok"
    if r["cut"]:
        status, out = "cut_off", ""
    elif not out:
        status = "empty_output"
    return {"output": out or fallback, "status": status, "llm_seconds": round(time.perf_counter() - t0, 2),
            "calls": 1, "tokens_in": r["tokens_in"], "tokens_out": r["tokens_out"]}


def run_jobs(jobs, work):
    path = RESULTS / "corrections" / f"{NAME}.csv"
    done = set()
    if path.exists():
        d = pd.read_csv(path, keep_default_na=False)
        d = d[~d.status.astype(str).str.startswith("error")]
        done = set(zip(d.clip_id, d.condition))
    todo = [j for j in jobs if (j["clip_id"], j["condition"]) not in done]
    print(f"{len(done)} outputs already saved, {len(todo)} to go")
    wall = 0.0
    for s in range(0, len(todo), CHUNK):
        chunk = todo[s:s + CHUNK]
        t0 = time.perf_counter()
        with ThreadPoolExecutor(WORKERS) as pool:
            results = list(pool.map(work, chunk))
        wall += time.perf_counter() - t0
        failed = sum(str(r["status"]).startswith("error") for r in results)
        if failed > len(chunk) // 2:
            raise RuntimeError(f"{failed} of {len(chunk)} calls failed ({results[0]['status']}). This chunk was not saved. "
                               "Fix the LLM, then run this notebook again: finished clips are skipped.")
        rows = [{"clip_id": j["clip_id"], "condition": j["condition"], **r} for j, r in zip(chunk, results)]
        pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False, encoding="utf-8")
        print(f"  {min(s + CHUNK, len(todo))}/{len(todo)} done, {failed} errors in this chunk, {wall / 60:.1f} min")
    return wall

HEAD = """You are a strict Urdu ASR post-editor. Correct ONLY clear recognition errors in the line: wrong or garbled words, wrongly split/merged words, obvious spelling mistakes.
Rules:
- Do NOT translate.
- Do NOT add, remove, reorder or rephrase words that are already correct.
- Do NOT add punctuation, diacritics or any explanation.
- If the line is already correct, return it unchanged.
Output ONLY the corrected Urdu line, nothing else."""


def example_block(stem):
    blocks = [f"Noisy: {prompt_text(getattr(r, stem))}\nCorrected: {prompt_text(r.reference)}" for r in examples.itertuples()]
    return "Here are 5 examples:\n\n" + "\n\n".join(blocks)


EXAMPLE_BLOCKS = {stem: example_block(stem) for stem in RECOGNISERS}


def build_prompt(stem, line):
    return f"{HEAD}\n\n{EXAMPLE_BLOCKS[stem]}\n\nNow correct this line. Output only the corrected line.\nNoisy: {line}\nCorrected:"

CONDITIONS = {stem: {"label": f"{name} + Gemini few-shot", "inputs": [stem]} for stem, name in RECOGNISERS.items()}

jobs = [{"clip_id": r.clip_id, "condition": stem, "input": prompt_text(getattr(r, stem))}
        for stem in RECOGNISERS for r in work_clips.itertuples()]


def work(job):
    if not job["input"]:
        return {"output": "", "status": "empty_input", "llm_seconds": 0, "calls": 0, "tokens_in": 0, "tokens_out": 0}
    return correct(build_prompt(job["condition"], job["input"]), job["input"])


wall = run_jobs(jobs, work)

0 outputs already saved, 7200 to go
  200/7200 done, 0 errors in this chunk, 0.5 min
  400/7200 done, 0 errors in this chunk, 0.9 min
  600/7200 done, 0 errors in this chunk, 1.3 min
  800/7200 done, 0 errors in this chunk, 1.6 min
  1000/7200 done, 0 errors in this chunk, 1.9 min
  1200/7200 done, 0 errors in this chunk, 2.2 min
  1400/7200 done, 0 errors in this chunk, 2.6 min
  1600/7200 done, 0 errors in this chunk, 3.0 min
  1800/7200 done, 0 errors in this chunk, 3.4 min
  2000/7200 done, 0 errors in this chunk, 3.8 min
  2200/7200 done, 0 errors in this chunk, 4.0 min
  2400/7200 done, 0 errors in this chunk, 4.3 min
  2600/7200 done, 0 errors in this chunk, 4.7 min
  2800/7200 done, 0 errors in this chunk, 5.0 min
  3000/7200 done, 0 errors in this chunk, 5.3 min
  3200/7200 done, 0 errors in this chunk, 5.6 min
  3400/7200 done, 0 errors in this chunk, 6.0 min
  3600/7200 done, 0 errors in this chunk, 6.4 min
  3800/7200 done, 0 errors in this chunk, 6.7 min
  4000/7200 done, 

## 4. Scores

In [4]:
from itertools import combinations

out = pd.read_csv(RESULTS / "corrections" / f"{NAME}.csv", keep_default_na=False)
have_ids = set.intersection(*[set(out[out.condition == c].clip_id) for c in CONDITIONS])
by_clip["reference_words"] = by_clip.reference_clean.str.split().str.len()
scored = by_clip.clip_id.isin(have_ids).values
scorable = (by_clip.reference_clean.str.len() > 0).values & scored
print(f"Clips with outputs for every condition: {scored.sum()} of {len(by_clip)}")


def edit_counts(ref, hyp):
    if not hyp:
        return 0, len(ref.split()), 0, len(ref.split()), len(ref), len(ref)
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return (w.substitutions, w.deletions, w.insertions, w.hits + w.substitutions + w.deletions,
            c.substitutions + c.deletions + c.insertions, c.hits + c.substitutions + c.deletions)


def counts_of(hyps):
    return np.array([edit_counts(r, h) if ok else (0, 0, 0, 0, 0, 0)
                     for r, h, ok in zip(by_clip.reference_clean, hyps, scorable)])


VERSIONS = {}
for stem, name in RECOGNISERS.items():
    VERSIONS[f"{name} (baseline)"] = {
        "hyp": list(by_clip[f"{stem}_clean"]), "recogniser_min_col": [f"{stem}_time_s"], "llm": None}
for cond, spec in CONDITIONS.items():
    o = out[out.condition == cond].drop_duplicates("clip_id", keep="last").set_index("clip_id").reindex(by_clip.clip_id)
    hyp = [clean_urdu(x) if isinstance(x, str) else "" for x in o.output.fillna("")]
    VERSIONS[spec["label"]] = {"hyp": hyp, "recogniser_min_col": [f"{s}_time_s" for s in spec["inputs"]],
                               "llm": o.reset_index(drop=True)}
for v in VERSIONS.values():
    v["counts"] = counts_of(v["hyp"])

is_fleurs, is_us = by_clip.dataset.eq("FLEURS"), by_clip.dataset.eq("UrduSpeech")
SUBSETS = [
    ("FLEURS", "test + validation", is_fleurs & by_clip.subset.isin(["test", "validation"])),
    ("FLEURS", "train", is_fleurs & by_clip.subset.eq("train")),
    ("FLEURS", "test + validation + train", is_fleurs),
    ("UrduSpeech", "benchmark (human-checked references)", is_us & by_clip.subset.eq("benchmark")),
    ("UrduSpeech", "corpus (Gemini references)", is_us & by_clip.subset.eq("corpus")),
    ("UrduSpeech", "benchmark + corpus", is_us),
]
FILTERS = [("all clips", scorable), ("references with 3+ words", scorable & (by_clip.reference_words >= 3).values)]

rows = []
for dataset, subset, in_subset in SUBSETS:
    for filter_name, keep in FILTERS:
        m = in_subset.values & keep
        if not m.any():
            continue
        for label, v in VERSIONS.items():
            c = v["counts"][m]
            words = c[:, 3].sum()
            row = {"dataset": dataset, "subset": subset, "clips_scored": filter_name, "clips": int(m.sum()),
                   "reference_words": int(words), "version": label,
                   "WER_%": round(100 * c[:, :3].sum() / words, 2), "CER_%": round(100 * c[:, 4].sum() / c[:, 5].sum(), 2),
                   "substitutions_%": round(100 * c[:, 0].sum() / words, 2),
                   "deletions_%": round(100 * c[:, 1].sum() / words, 2),
                   "insertions_%": round(100 * c[:, 2].sum() / words, 2),
                   "recogniser_min": round(by_clip.loc[m, v["recogniser_min_col"]].astype(float).sum().sum() / 60, 1)}
            if v["llm"] is not None:
                l = v["llm"][m]
                row.update({"llm_min": round(l.llm_seconds.astype(float).sum() / 60, 1), "llm_calls": int(l.calls.astype(int).sum()),
                            "tokens_in": int(l.tokens_in.astype(int).sum()), "tokens_out": int(l.tokens_out.astype(int).sum()),
                            "errors_or_cut_off": int((l.status.astype(str).str.startswith("error") | l.status.eq("cut_off")).sum())})
            rows.append(row)
scores = pd.DataFrame(rows)
scores.to_csv(RESULTS / "scores" / f"{NAME}_scores.csv", index=False, encoding="utf-8-sig")


def paired_bootstrap(errors_a, errors_b, words, n=1000, seed=42):
    idx = np.random.default_rng(seed).integers(0, len(words), size=(n, len(words)))
    diff = 100 * (errors_a[idx].sum(1) - errors_b[idx].sum(1)) / words[idx].sum(1)
    return np.percentile(diff, [2.5, 97.5])


rows = []
for dataset, subset, in_subset in SUBSETS:
    for filter_name, keep in FILTERS:
        m = in_subset.values & keep
        if not m.any():
            continue
        for cond, spec in CONDITIONS.items():
            a = VERSIONS[spec["label"]]
            for stem in spec["inputs"]:
                b_label = f"{RECOGNISERS[stem]} (baseline)"
                b = VERSIONS[b_label]
                ea, eb = a["counts"][m, :3].sum(1), b["counts"][m, :3].sum(1)
                words = a["counts"][m, 3]
                lo, hi = paired_bootstrap(ea, eb, words)
                wa, wb = 100 * ea.sum() / words.sum(), 100 * eb.sum() / words.sum()
                changed = np.array([x != y for x, y in zip(np.array(a["hyp"], dtype=object)[m], np.array(b["hyp"], dtype=object)[m])])
                rows.append({"dataset": dataset, "subset": subset, "clips_scored": filter_name, "clips": int(m.sum()),
                             "corrected": spec["label"], "baseline": b_label,
                             "WER_corrected_%": round(wa, 2), "WER_baseline_%": round(wb, 2),
                             "difference": round(wa - wb, 2), "interval_low": round(lo, 2), "interval_high": round(hi, 2),
                             "result": "corrected lower" if hi < 0 else "baseline lower" if lo > 0 else "no reliable difference",
                             "lines_changed_%": round(100 * changed.mean(), 1),
                             "lines_improved": int(((ea < eb) & changed).sum()), "lines_worsened": int(((ea > eb) & changed).sum())})
vs = pd.DataFrame(rows)
vs.to_csv(RESULTS / "scores" / f"{NAME}_vs_baseline.csv", index=False, encoding="utf-8-sig")

shown = scores[scores.clips_scored == "all clips"].pivot_table(index=["dataset", "subset"], columns="version", values="WER_%", sort=False)
print("WER % on all clips")
print(shown.to_string())
print("\nDifference to the baseline recogniser (corrected minus baseline, 95% interval), all clips")
print(vs[vs.clips_scored == "all clips"][["subset", "corrected", "baseline", "difference", "interval_low", "interval_high", "result"]].to_string(index=False))

Clips with outputs for every condition: 2400 of 2400
WER % on all clips
version                                          Whisper-large-v3 (baseline)  Omnilingual LLM 1B v2 (baseline)  Indic Transcribe Core (baseline)  Whisper-large-v3 + Gemini few-shot  Omnilingual LLM 1B v2 + Gemini few-shot  Indic Transcribe Core + Gemini few-shot
dataset    subset                                                                                                                                                                                                                                                     
FLEURS     test + validation                                           20.64                             15.24                             12.57                               14.54                                    13.04                                    12.62
           train                                                       18.72                              6.58                             

## 5. Run information

In [5]:
info_path = RESULTS / "run_info" / f"{NAME}.json"
previous = json.loads(info_path.read_text()) if info_path.exists() else {}
info = {
    "notebook": NAME,
    "llm": LLM_NAME,
    "workers": WORKERS,
    "prompt": "gec_few_shot",
    "temperature": 0.0, "max_tokens": 512,
    "example_clips": list(examples.clip_id),
    "machine": {"python": platform.python_version(), "platform": platform.platform()},
    "packages": {p: importlib.metadata.version(p) for p in ("pandas", "numpy", "jiwer")},
    "normaliser": "clean_urdu v1",
    "bootstrap": {"resamples": 1000, "seed": 42},
    "llm_wall_minutes": round(previous.get("llm_wall_minutes", 0) + wall / 60, 1),
}
info_path.write_text(json.dumps(info, indent=2, ensure_ascii=False))
print(json.dumps(info, indent=2, ensure_ascii=False))

{
  "notebook": "gec_few_shot_gemini",
  "llm": "google/gemini-3.1-flash-lite",
  "workers": 12,
  "prompt": "gec_few_shot",
  "temperature": 0.0,
  "max_tokens": 512,
  "example_clips": [
    "fleurs/test/17944002793012079254",
    "fleurs/test/1144630914630122970",
    "fleurs/test/5871438281700834673",
    "fleurs/test/1136035778246969234",
    "fleurs/validation/8466778121606878841"
  ],
  "machine": {
    "python": "3.10.12",
    "platform": "Linux-6.18.40.1-microsoft-standard-WSL2-x86_64-with-glibc2.35"
  },
  "packages": {
    "pandas": "2.2.3",
    "numpy": "1.26.4",
    "jiwer": "4.0.0"
  },
  "normaliser": "clean_urdu v1",
  "bootstrap": {
    "resamples": 1000,
    "seed": 42
  },
  "llm_wall_minutes": 12.5
}
